# Scenario: File Processors API (RHAIENG-5114 regression coverage)

**QE Perspective:** The `file_processors` API was missing from the OGX distro config, breaking file_search / vector store file ingestion. We validate that:

- The `file_processors` provider is registered (pypdf or auto).
- **Direct upload** processing returns chunks with content and metadata.
- **file_id-based** processing works for files already uploaded via `/v1/files`.
- Chunk metadata references the source file correctly.

This is a v1alpha endpoint (`/v1alpha/file-processors/process`) not covered by the OGX Python SDK, so we use `httpx` directly.

## Setup

Load base URL from environment; create httpx client. MODEL is not needed for file processing.

In [ ]:
import os
import httpx

base_url = os.environ.get("BASE_URL", "http://localhost:8321")
base_url = base_url.rstrip("/")

assert base_url, "BASE_URL must be set"

client = httpx.Client(base_url=base_url, timeout=30.0)
print(f"Base URL: {base_url}")

## Verify file_processors provider is registered

The fix for RHAIENG-5114 adds a `file_processors` provider to the distro config. Verify it exists.

In [ ]:
resp = client.get("/v1/providers")
assert resp.status_code == 200, f"GET /v1/providers failed: {resp.status_code}"

providers = resp.json()["data"]
fp_providers = [p for p in providers if p.get("api") == "file_processors"]

assert len(fp_providers) > 0, (
    "No file_processors provider registered — RHAIENG-5114 regression: "
    "file_processors API is missing from distro config"
)

for p in fp_providers:
    print(f"  {p['provider_id']} ({p['provider_type']})")

## Test 1: Direct file upload processing

Upload a text file directly to `/v1alpha/file-processors/process` and verify chunks are returned.

In [ ]:
test_content = b"Hello from file_processors notebook test"

resp = client.post(
    "/v1alpha/file-processors/process",
    files={"file": ("test.txt", test_content, "text/plain")},
)
assert resp.status_code == 200, (
    f"Direct processing failed: {resp.status_code} {resp.text}"
)

body = resp.json()
assert "chunks" in body, "Response missing 'chunks' field"
assert isinstance(body["chunks"], list), "chunks should be a list"
assert len(body["chunks"]) >= 1, "Expected at least one chunk"

chunk = body["chunks"][0]
assert "content" in chunk, "Chunk missing 'content' field"
assert isinstance(chunk["content"], str), "Chunk content should be a string"
assert len(chunk["content"]) > 0, "Chunk content should not be empty"

assert "metadata" in body, "Response missing 'metadata' field"
assert "processor" in body["metadata"], "Metadata missing 'processor' field"

print(f"Chunks: {len(body['chunks'])}")
print(f"Content: {chunk['content']!r}")
print(f"Processor: {body['metadata']['processor']}")

## Test 2: Process file by file_id

Upload a file via `/v1/files`, then process it by referencing the `file_id`. This is the path used by vector store file ingestion (the flow broken by RHAIENG-5114).

In [ ]:
upload_resp = client.post(
    "/v1/files",
    files={"file": ("test-fp.txt", test_content, "text/plain")},
    data={"purpose": "assistants"},
)
assert upload_resp.status_code in (200, 201), (
    f"File upload failed: {upload_resp.status_code}"
)

file_id = upload_resp.json()["id"]
assert file_id, "File upload did not return an ID"
print(f"Uploaded file_id: {file_id}")

proc_resp = client.post(
    "/v1alpha/file-processors/process",
    data={"file_id": file_id},
)
assert proc_resp.status_code == 200, (
    f"file_id processing failed: {proc_resp.status_code} {proc_resp.text}"
)

body = proc_resp.json()
assert len(body["chunks"]) >= 1, "Expected at least one chunk"

chunk = body["chunks"][0]
assert chunk["content"], "Chunk content should not be empty"

chunk_meta = chunk.get("metadata", {})
assert chunk_meta.get("file_id") == file_id, (
    f"Chunk metadata file_id mismatch: expected {file_id}, got {chunk_meta.get('file_id')}"
)

print(f"Chunks: {len(body['chunks'])}")
print(f"Content: {chunk['content']!r}")
print(f"Chunk file_id: {chunk_meta.get('file_id')}")

## Test 3: Owner-encrypted PDF processing (RHAIENG-5857)

Owner-encrypted PDFs restrict printing/copying but require no password to read.
Upstream [ogx-ai/ogx#5670](https://github.com/ogx-ai/ogx/pull/5670) added a blanket `is_encrypted` check
that rejects these. This test catches that regression.

In [ ]:
from pathlib import Path

pdf_path = Path("../../tests/fixtures/sample-encrypted.pdf")
assert pdf_path.exists(), f"Fixture not found: {pdf_path}"

with open(pdf_path, "rb") as f:
    resp = client.post(
        "/v1alpha/file-processors/process",
        files={"file": ("sample-encrypted.pdf", f, "application/pdf")},
    )

assert resp.status_code == 200, (
    f"Server rejected owner-encrypted PDF (HTTP {resp.status_code}). "
    f"Regression: blanket is_encrypted check — see ogx-ai/ogx#6181"
)

data = resp.json()
chunks = data.get("chunks", [])
assert len(chunks) > 0, "No chunks returned from encrypted PDF"

marker = "OGX-RAG-SMOKE-7f3a9c2e8b1d4f06"
all_text = " ".join(c.get("content", "") for c in chunks)
assert marker in all_text, f"Expected marker '{marker}' not found in extracted text"

print(f"Owner-encrypted PDF: {len(chunks)} chunks, marker found")

## Cleanup

Delete the uploaded file.

In [ ]:
if "file_id" in dir():
    del_resp = client.delete(f"/v1/files/{file_id}")
    assert del_resp.status_code in (200, 204), (
        f"File delete failed: {del_resp.status_code}"
    )
    print(f"Deleted file {file_id}")

client.close()

## QE Assertions summary

- **Provider registered**: `file_processors` API has at least one provider (regression gate for RHAIENG-5114).
- **Direct upload**: processing returns chunks with non-empty content and processor metadata.
- **file_id processing**: uploaded file can be processed by ID; chunk metadata references the correct file_id.
- **Cleanup**: uploaded file is deleted successfully.